<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Video_PostProcessor_Director_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

View this notebook on GitHub: <a href="https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/Video_PostProcessor_Director_Colab.ipynb" target="_parent">Skquark/AEI-Colab-Notebooks/Video_PostProcessor_Director_Colab.ipynb</a>


# 🎬 AEI Director Studio — Multi-Modal Timeline Assembler, Auto-Ducking 4-Stem Audio Mixer & Word-Highlighted Subtitle Studio

A unified **Post-Production, Scoring, Subtitling & Timeline Assembly Studio** designed as the finishing hub for the entire 65+ notebook AEI ecosystem. It automatically discovers your generated assets across `/content/drive/MyDrive/AEI_3D_Out/` and `/content/drive/MyDrive/AEI_TTS_Out/` and turns raw clips into mastered, broadcast-ready videos:

- **Video & Avatar Clips**: [`Wan2.2`](https://github.com/Wan-Video/Wan2.2), [`Wan-VACE`](https://github.com/ali-vilab/VACE), [`FramePack`](https://github.com/lllyasviel/FramePack), [`LTX-Video-2.5`](https://github.com/Lightricks/LTX-Video), [`MiniMax-H3`](https://huggingface.co/MiniMaxAI), [`InfiniteTalk`](https://github.com/MeiGen-AI/InfiniteTalk), [`LongCat-Video-Avatar-1.5`](https://github.com/meituan-longcat/LongCat-Video), [`LivePortrait`](https://github.com/KwaiVGI/LivePortrait), [`SeedVR2`](https://github.com/ByteDance-Seed/SeedVR2)
- **Dialogue & Voiceover Stems**: [`Ming-3.0-TTS`](https://github.com/inclusionAI/Ming), [`Qwen3-TTS`](https://github.com/QwenLM/Qwen3-TTS), [`VoxCPM2`](https://github.com/OpenBMB/VoxCPM), [`MOSS-TTS`](https://github.com/OpenMOSS/MOSS-TTS), [`Fish-S2-Pro`](https://github.com/fishaudio/fish-speech), [`Chatterbox`](https://github.com/resemble-ai/chatterbox), [`Kokoro-82M`](https://github.com/hexgrad/kokoro), [`Dia`](https://github.com/nari-labs/dia), [`Seed-VC`](https://github.com/Plachtaa/seed-vc)
- **Synchronized Foley & Sound Effects**: [`HunyuanVideo-Foley`](https://github.com/Tencent-Hunyuan/HunyuanVideo-Foley), [`MMAudio`](https://github.com/hkchengrex/MMAudio), [`AudioX-Turbo`](https://github.com/NoizAI/AudioX-Turbo), [`Stable-Audio-3`](https://github.com/Stability-AI/stable-audio-3), [`MOSS-SoundEffect-v2.0`](https://huggingface.co/OpenMOSS-Team/MOSS-SoundEffect-v2.0), [`MiDashengLM-Gen`](https://github.com/xiaomi-research/midashenglm-gen), [`TangoFlux-EzAudio`](https://github.com/declare-lab/TangoFlux)
- **Background Music (BGM)**: [`ACE-Step`](https://github.com/ace-step/ACE-Step), [`MiniMax-Music3`](https://huggingface.co/MiniMaxAI/MiniMax-Music3)

---

### Architecture & Core Capabilities

```
┌───────────────────────────────────────────────────────────────────────────────────┐
│                      AEI Director Studio (4-Stage Pipeline)                       │
├───────────────────────────────────────────────────────────────────────────────────┤
│ 1. 4-Stem Audio Mixer & Sidechain Speech Ducking (48 kHz Stereo Float32)          │
│    • Stems: [Video Audio] + [Voiceover/TTS] + [Foley/SFX (±ms)] + [BGM Loop/Trim] │
│    • Vocal-Band RMS Envelope Follower -> Smooth Attack/Hold/Release BGM Ducking   │
│    • Master Bus: 40 Hz Butterworth HPF + ITU-R BS.1770-4 (-14 / -16 LUFS) Limiter │
├───────────────────────────────────────────────────────────────────────────────────┤
│ 2. Word-Level ASR & Karaoke Subtitle Engine (faster-whisper + stable-ts + libass) │
│    • Silero VAD + stabilized word boundaries -> Editable Subtitle Table           │
│    • 6 ASS Styles: Viral Karaoke {\k} Word Pop, Cinema Classic, Yellow Doc, etc.  │
├───────────────────────────────────────────────────────────────────────────────────┤
│ 3. Multi-Clip Timeline Sequencer, xfade Transitions & Aspect-Ratio Finisher       │
│    • Canvas: 16:9 1080p, 9:16 Vertical (Blurred Glass Backdrop / Crop), 1:1, 4K   │
│    • Transitions: Cut, Fade, Dissolve, WipeLeft/Right, SlideUp, CircleOpen        │
│    • 6 Color Grades: Neutral, Teal & Orange, Golden Hour, Kodak Film, Cyberpunk   │
└───────────────────────────────────────────────────────────────────────────────────┘
```

| Component | Package / Model | License | Compute |
| :--- | :--- | :--- | :--- |
| **Word-Level ASR & Alignment** | [`faster-whisper`](https://github.com/SYSTRAN/faster-whisper) (`large-v3-turbo` / `base` / `small`) + [`stable-ts`](https://github.com/jianfch/stable-ts) | MIT | T4 GPU (`float16`, ~15× real-time) or CPU (`int8`, ~3× real-time) |
| **DSP Mixing & LUFS Mastering** | `scipy.signal` + `soundfile` + [`pyloudnorm`](https://github.com/csteinmetz1/pyloudnorm) + [`pedalboard`](https://github.com/spotify/pedalboard) | MIT / BSD / GPLv3 | CPU (instantaneous) |
| **Subtitle & Video Compositing** | `ffmpeg` (`libass`, `xfade`, `acrossfade`, `colorbalance`, `eq`) | LGPL / GPL | CPU / NVENC |


In [ ]:
# @title STEP 1 — Install Dependencies & Mount Google Drive
# @markdown Installs `faster-whisper`, `stable-ts`, `pyloudnorm`, `pedalboard`, `soundfile`, `librosa`, `scipy`, `matplotlib`, and `gradio==5.49.1`, verifies `ffmpeg` (`libass` + `xfade`), and mounts Google Drive.

import os, sys, time, subprocess, shutil, pathlib

t0 = time.time()
print("=" * 72)
print("🎬 AEI Director Studio — Step 1: Environment & Dependency Setup")
print("=" * 72)

MOUNT_DRIVE = True  # @param {type:"boolean"}
DRIVE_CACHE_DIR = "/content/drive/MyDrive/AEI_Models_Cache/Director_Studio"  # @param {type:"string"}
OUTPUT_BASE_DIR = "/content/drive/MyDrive/AEI_3D_Out/Director_Studio"  # @param {type:"string"}

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        if not os.path.isdir("/content/drive/MyDrive"):
            drive.mount("/content/drive", force_remount=False)
        print("[Drive] Mounted at /content/drive/MyDrive")
    except ImportError:
        print("[Drive] Not running on Google Colab — using local paths.")
        DRIVE_CACHE_DIR = os.path.abspath("./_director_cache")
        OUTPUT_BASE_DIR = os.path.abspath("./_director_outputs")
    except Exception as e:
        print(f"[Drive] Mount warning ({e}) — falling back to /content/director_outputs")
        DRIVE_CACHE_DIR = "/content/_director_cache"
        OUTPUT_BASE_DIR = "/content/director_outputs"
else:
    DRIVE_CACHE_DIR = "/content/_director_cache"
    OUTPUT_BASE_DIR = "/content/director_outputs"

os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)
os.environ["HF_HOME"] = os.path.join(DRIVE_CACHE_DIR, "hf_home")
os.environ["HUGGINGFACE_HUB_CACHE"] = os.path.join(DRIVE_CACHE_DIR, "hf_home", "hub")
os.makedirs(os.environ["HUGGINGFACE_HUB_CACHE"], exist_ok=True)

# Ensure system ffmpeg + fonts are installed for libass subtitle rendering
try:
    subprocess.run(["apt-get", "update", "-qq"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "ffmpeg", "libass9", "fonts-dejavu-core", "fonts-liberation", "fonts-noto-core"],
        check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
except Exception as e:
    print(f"[apt] System font/ffmpeg check skipped: {e}")

pkgs = [
    "faster-whisper>=1.1.0",
    "stable-ts>=2.17.4",
    "ctranslate2>=4.4.0",
    "pyloudnorm",
    "pedalboard",
    "soundfile",
    "librosa",
    "scipy",
    "matplotlib",
    "pydub",
    "imageio-ffmpeg",
    "huggingface_hub",
    "tqdm==4.66.5",
    "gradio==5.49.1",
]

print("[pip] Installing audio/video post-production & ASR packages...")
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet"] + pkgs, check=True)
except Exception as e:
    print(f"[pip] Warning during batch install ({e}); retrying core packages individually...")
    for p in pkgs:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", p], check=False)

import torch
import soundfile as sf
import pyloudnorm as pyln
import faster_whisper
import gradio as gr

try:
    import stable_whisper
    _HAVE_STABLE_TS = True
except Exception as e:
    print(f"[warn] stable_whisper import note: {e}")
    _HAVE_STABLE_TS = False

print(f"[OK] Setup finished in {time.time() - t0:.1f}s")
print(f"     PyTorch        : {torch.__version__} (CUDA available: {torch.cuda.is_available()})")
print(f"     faster-whisper : {faster_whisper.__version__} | stable-ts: {_HAVE_STABLE_TS}")
print(f"     Gradio         : {gr.__version__}")
print(f"     HF_HOME Cache  : {os.environ['HF_HOME']}")
print(f"     Output Folder  : {OUTPUT_BASE_DIR}")


In [ ]:
# @title STEP 2 — Pre-Cache Whisper ASR Weights & Scan Cross-Notebook Drive Library
# @markdown Downloads the selected `faster-whisper` CTranslate2 checkpoint (`large-v3-turbo`, `small`, or `base`) to your Google Drive cache (`HF_HOME`) and indexes all video, TTS/dialogue, foley/SFX, and BGM assets across your Drive `AEI_3D_Out` and `AEI_TTS_Out` folders.

import os, glob, time
import torch

WHISPER_MODEL_SIZE = "deepdml/faster-whisper-large-v3-turbo-ct2"  # @param ["deepdml/faster-whisper-large-v3-turbo-ct2", "large-v3", "medium", "small", "base", "tiny"]
PRELOAD_ASR_ON_STARTUP = False  # @param {type:"boolean"}

ASR_MODEL_CACHE = {"model": None, "model_id": None, "backend": None}

def load_asr_engine(model_id: str = WHISPER_MODEL_SIZE, force_reload: bool = False):
    """Lazy loader for faster-whisper / stable-ts with CTranslate2 GPU/CPU auto-selection."""
    global ASR_MODEL_CACHE
    if not force_reload and ASR_MODEL_CACHE["model"] is not None and ASR_MODEL_CACHE["model_id"] == model_id:
        return ASR_MODEL_CACHE["model"], ASR_MODEL_CACHE["backend"]

    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"
    print(f"[ASR] Loading '{model_id}' on {device} ({compute_type})...")
    t0 = time.time()

    try:
        import stable_whisper
        model = stable_whisper.load_faster_whisper(
            model_id,
            device=device,
            compute_type=compute_type,
            download_root=os.environ.get("HUGGINGFACE_HUB_CACHE"),
        )
        backend = "stable_faster_whisper"
    except Exception as e:
        print(f"[ASR] stable_whisper wrapper fallback ({e}); using faster_whisper.WhisperModel directly...")
        from faster_whisper import WhisperModel
        try:
            model = WhisperModel(
                model_id,
                device=device,
                compute_type=compute_type,
                download_root=os.environ.get("HUGGINGFACE_HUB_CACHE"),
            )
        except Exception:
            model = WhisperModel(
                "small",
                device=device,
                compute_type=compute_type,
                download_root=os.environ.get("HUGGINGFACE_HUB_CACHE"),
            )
        backend = "faster_whisper"

    ASR_MODEL_CACHE = {"model": model, "model_id": model_id, "backend": backend}
    print(f"[ASR] Loaded '{model_id}' ({backend}) in {time.time() - t0:.1f}s")
    return model, backend

def scan_drive_media_assets(max_per_category: int = 80) -> dict:
    """Scans AEI_3D_Out and AEI_TTS_Out on Google Drive to categorize generated videos, dialogue, foley, and BGM."""
    roots = [
        "/content/drive/MyDrive/AEI_3D_Out",
        "/content/drive/MyDrive/AEI_TTS_Out",
        OUTPUT_BASE_DIR,
    ]
    videos, dialogues, foleys, bgms = [], [], [], []
    video_exts = {".mp4", ".mov", ".webm", ".mkv"}
    audio_exts = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".opus"}

    foley_keywords = ("foley", "sfx", "mmaudio", "audiox", "stable-audio", "moss-soundeffect", "midasheng", "tangoflux", "ezaudio")
    bgm_keywords = ("music", "ace-step", "ace_step", "minimax-music", "song", "bgm")

    seen = set()
    for r in roots:
        if not os.path.isdir(r):
            continue
        try:
            for dirpath, _, filenames in os.walk(r):
                if "/_thumbs" in dirpath or "\\_thumbs" in dirpath:
                    continue
                for fn in filenames:
                    full = os.path.join(dirpath, fn)
                    if full in seen:
                        continue
                    seen.add(full)
                    ext = os.path.splitext(fn)[1].lower()
                    low_path = full.lower()
                    try:
                        mtime = os.path.getmtime(full)
                    except OSError:
                        mtime = 0.0
                    if ext in video_exts:
                        videos.append((mtime, full))
                    elif ext in audio_exts:
                        if any(k in low_path for k in bgm_keywords):
                            bgms.append((mtime, full))
                        elif any(k in low_path for k in foley_keywords):
                            foleys.append((mtime, full))
                        else:
                            dialogues.append((mtime, full))
        except Exception as e:
            print(f"[Scan] Warning scanning {r}: {e}")

    videos.sort(reverse=True)
    dialogues.sort(reverse=True)
    foleys.sort(reverse=True)
    bgms.sort(reverse=True)

    return {
        "videos": [p for _, p in videos[:max_per_category]],
        "dialogues": [p for _, p in dialogues[:max_per_category]],
        "foleys": [p for _, p in foleys[:max_per_category]],
        "bgms": [p for _, p in bgms[:max_per_category]],
    }

DRIVE_ASSETS = scan_drive_media_assets()
print(f"[Library Scan] Discovered across Drive:")
print(f"  • Video Clips          : {len(DRIVE_ASSETS['videos'])}")
print(f"  • Dialogue / TTS Stems : {len(DRIVE_ASSETS['dialogues'])}")
print(f"  • Foley / SFX Stems    : {len(DRIVE_ASSETS['foleys'])}")
print(f"  • Music / BGM Stems    : {len(DRIVE_ASSETS['bgms'])}")

if PRELOAD_ASR_ON_STARTUP:
    try:
        load_asr_engine(WHISPER_MODEL_SIZE)
    except Exception as e:
        print(f"[ASR] Preload deferred until first transcription call: {e}")


In [ ]:
# @title STEP 3 — Core Post-Production Engine (4-Stem Auto-Ducking Mixer, Karaoke ASS Subtitler & Timeline Sequencer)
# @markdown Defines the DSP sidechain speech-ducking mixer, ITU-R BS.1770-4 loudness normalizer, `faster-whisper` + `stable-ts` word-level subtitle generator, `.ass` karaoke renderer, and multi-clip `ffmpeg xfade` timeline assembler.

import os, sys, json, time, math, shutil, tempfile, subprocess
from pathlib import Path
import numpy as np
import scipy.signal as sps
import soundfile as sf
import pyloudnorm as pyln
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

TARGET_SR = 48000

# ── 1. Media Probing & Safe Audio Extraction ─────────────────────────────────
def probe_video_info(video_path: str) -> dict:
    """Returns duration_sec, width, height, fps, and has_audio using ffprobe."""
    info = {"duration": 5.0, "width": 1280, "height": 720, "fps": 30.0, "has_audio": False}
    if not video_path or not os.path.isfile(video_path):
        return info
    try:
        cmd = [
            "ffprobe", "-v", "error", "-show_entries",
            "format=duration:stream=index,codec_type,width,height,r_frame_rate",
            "-of", "json", video_path
        ]
        res = subprocess.run(cmd, capture_output=True, text=True, timeout=20, check=False)
        data = json.loads(res.stdout or "{}")
        fmt_dur = data.get("format", {}).get("duration")
        if fmt_dur:
            info["duration"] = max(0.2, float(fmt_dur))
        for st in data.get("streams", []):
            ctype = st.get("codec_type")
            if ctype == "video" and st.get("width"):
                info["width"] = int(st["width"])
                info["height"] = int(st["height"])
                r_fps = st.get("r_frame_rate", "30/1")
                if "/" in r_fps:
                    num, den = r_fps.split("/", 1)
                    if float(den) > 0:
                        info["fps"] = round(float(num) / float(den), 3)
            elif ctype == "audio":
                info["has_audio"] = True
    except Exception as e:
        print(f"[probe_video_info] Warning: {e}")
    return info

def load_audio_stereo_48k(path: str, target_sr: int = TARGET_SR) -> np.ndarray | None:
    """Loads any audio or video file into a (N, 2) float32 numpy array at target_sr."""
    if not path or not os.path.isfile(path):
        return None
    tmp_wav = tempfile.mktemp(suffix=".wav")
    try:
        cmd = [
            "ffmpeg", "-y", "-v", "error", "-i", path,
            "-vn", "-ac", "2", "-ar", str(target_sr), "-c:a", "pcm_f32le", tmp_wav
        ]
        r = subprocess.run(cmd, capture_output=True, text=True, check=False)
        if r.returncode != 0 or not os.path.isfile(tmp_wav):
            return None
        audio, sr = sf.read(tmp_wav, dtype="float32", always_2d=True)
        if audio.shape[1] == 1:
            audio = np.repeat(audio, 2, axis=1)
        elif audio.shape[1] > 2:
            audio = audio[:, :2]
        return audio
    except Exception as e:
        print(f"[load_audio_stereo_48k] Warning loading {path}: {e}")
        return None
    finally:
        if os.path.isfile(tmp_wav):
            try:
                os.remove(tmp_wav)
            except OSError:
                pass

def place_stem_on_timeline(
    stem: np.ndarray | None,
    total_samples: int,
    offset_ms: float = 0.0,
    gain_db: float = 0.0,
    loop_to_fill: bool = False,
    fade_in_ms: float = 15.0,
    fade_out_ms: float = 80.0,
    sr: int = TARGET_SR,
) -> np.ndarray:
    """Places a stereo stem onto a fixed-length timeline with offset, gain, optional crossfade looping, and fades."""
    canvas = np.zeros((total_samples, 2), dtype=np.float32)
    if stem is None or len(stem) == 0:
        return canvas

    gain = float(10.0 ** (gain_db / 20.0))
    work = stem.astype(np.float32) * gain

    if loop_to_fill and len(work) < total_samples:
        xfade_samples = min(int(sr * 1.5), len(work) // 4)
        out = work.copy()
        while len(out) < total_samples + int(abs(offset_ms) * sr / 1000.0):
            if xfade_samples > 64:
                fade_out = np.linspace(1.0, 0.0, xfade_samples, dtype=np.float32)[:, None]
                fade_in = np.linspace(0.0, 1.0, xfade_samples, dtype=np.float32)[:, None]
                out[-xfade_samples:] = out[-xfade_samples:] * fade_out + work[:xfade_samples] * fade_in
                out = np.vstack([out, work[xfade_samples:]])
            else:
                out = np.vstack([out, work])
        work = out

    offset_samples = int(round((offset_ms / 1000.0) * sr))
    if offset_samples >= 0:
        dst_start = min(total_samples, offset_samples)
        copy_len = min(total_samples - dst_start, len(work))
        if copy_len > 0:
            canvas[dst_start:dst_start + copy_len] = work[:copy_len]
    else:
        src_start = min(len(work), -offset_samples)
        copy_len = min(total_samples, len(work) - src_start)
        if copy_len > 0:
            canvas[:copy_len] = work[src_start:src_start + copy_len]

    # Apply subtle fade-in and fade-out
    fi = min(int(sr * fade_in_ms / 1000.0), total_samples // 4)
    fo = min(int(sr * fade_out_ms / 1000.0), total_samples // 4)
    if fi > 1:
        canvas[:fi] *= np.linspace(0.0, 1.0, fi, dtype=np.float32)[:, None]
    if fo > 1:
        canvas[-fo:] *= np.linspace(1.0, 0.0, fo, dtype=np.float32)[:, None]
    return canvas

# ── 2. Sidechain Speech-Activity Envelope Ducking ─────────────────────────────
def compute_speech_ducking_envelope(
    speech_bus: np.ndarray,
    sr: int = TARGET_SR,
    threshold_db: float = -36.0,
    duck_depth_db: float = -12.0,
    attack_ms: float = 45.0,
    hold_ms: float = 250.0,
    release_ms: float = 650.0,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Computes a smooth sample-accurate gain multiplier (0..1) and dB reduction curve
    that ducks BGM whenever vocal-band (120 Hz - 4200 Hz) energy exceeds threshold_db.
    """
    n_samples = len(speech_bus)
    if n_samples == 0 or abs(duck_depth_db) < 0.1:
        return np.ones((n_samples, 1), dtype=np.float32), np.zeros(n_samples, dtype=np.float32)

    mono = np.mean(speech_bus, axis=1)
    try:
        sos = sps.butter(2, [120.0, 4200.0], btype="bandpass", fs=sr, output="sos")
        vocal_band = sps.sosfiltfilt(sos, mono)
    except Exception:
        vocal_band = mono

    hop = max(1, int(sr * 0.010))  # 10 ms frame step
    win = max(hop, int(sr * 0.030))
    n_frames = int(math.ceil(n_samples / hop))
    rms_db = np.full(n_frames, -90.0, dtype=np.float32)

    for i in range(n_frames):
        s = i * hop
        e = min(n_samples, s + win)
        chunk = vocal_band[s:e]
        if len(chunk) > 0:
            r = float(np.sqrt(np.mean(chunk * chunk) + 1e-12))
            rms_db[i] = 20.0 * math.log10(max(r, 1e-6))

    active = rms_db > threshold_db
    # Apply hold frames so brief inter-word pauses don't cause BGM pumping
    hold_frames = max(1, int(round(hold_ms / 10.0)))
    held_active = active.copy()
    last_active = -999999
    for i in range(n_frames):
        if active[i]:
            last_active = i
        elif (i - last_active) <= hold_frames:
            held_active[i] = True

    target_db = np.where(held_active, -abs(duck_depth_db), 0.0).astype(np.float32)
    env_db_frames = np.zeros(n_frames, dtype=np.float32)

    alpha_att = math.exp(-1.0 / max(1.0, (attack_ms / 10.0)))
    alpha_rel = math.exp(-1.0 / max(1.0, (release_ms / 10.0)))

    cur = 0.0
    for i in range(n_frames):
        tgt = float(target_db[i])
        if tgt < cur:
            cur = alpha_att * cur + (1.0 - alpha_att) * tgt
        else:
            cur = alpha_rel * cur + (1.0 - alpha_rel) * tgt
        env_db_frames[i] = cur

    frame_centers = np.minimum(np.arange(n_frames) * hop + hop // 2, n_samples - 1)
    env_db_samples = np.interp(np.arange(n_samples), frame_centers, env_db_frames).astype(np.float32)
    gain_linear = (10.0 ** (env_db_samples / 20.0)).astype(np.float32)[:, None]
    return gain_linear, env_db_samples

# ── 3. Complete 4-Stem Scene Scorer & Master Bus ──────────────────────────────
def mix_and_master_scene(
    video_path: str,
    vo_path: str | None = None,
    foley_path: str | None = None,
    bgm_path: str | None = None,
    orig_gain_db: float = 0.0,
    mute_orig_audio: bool = False,
    vo_gain_db: float = 0.0,
    vo_offset_ms: float = 0.0,
    foley_gain_db: float = 0.0,
    foley_offset_ms: float = 0.0,
    bgm_gain_db: float = -8.0,
    auto_duck_bgm: bool = True,
    duck_threshold_db: float = -36.0,
    duck_depth_db: float = -12.0,
    duck_attack_ms: float = 45.0,
    duck_hold_ms: float = 250.0,
    duck_release_ms: float = 650.0,
    master_lufs: float = -14.0,
    true_peak_db: float = -1.0,
    output_dir: str = OUTPUT_BASE_DIR,
) -> dict:
    """
    Mixes up to 4 stereo audio stems against `video_path`, applies sidechain speech ducking
    to BGM, masters the stereo bus to ITU-R BS.1770-4 LUFS, muxes the finished MP4,
    and generates a dark-mode waveform & ducking diagnostic plot.
    """
    if not video_path or not os.path.isfile(video_path):
        raise ValueError("Please provide or select a valid input video file (.mp4/.mov/.webm).")

    os.makedirs(output_dir, exist_ok=True)
    vinfo = probe_video_info(video_path)
    dur_s = max(0.5, vinfo["duration"])
    total_samples = int(round(dur_s * TARGET_SR))

    orig_raw = None if mute_orig_audio else load_audio_stereo_48k(video_path, TARGET_SR)
    vo_raw = load_audio_stereo_48k(vo_path, TARGET_SR) if vo_path else None
    foley_raw = load_audio_stereo_48k(foley_path, TARGET_SR) if foley_path else None
    bgm_raw = load_audio_stereo_48k(bgm_path, TARGET_SR) if bgm_path else None

    # Apply 80 Hz rumble filter to Voiceover if present
    if vo_raw is not None and len(vo_raw) > 128:
        try:
            sos_vo = sps.butter(2, 80.0, btype="highpass", fs=TARGET_SR, output="sos")
            vo_raw = sps.sosfiltfilt(sos_vo, vo_raw, axis=0).astype(np.float32)
        except Exception:
            pass

    stem_orig = place_stem_on_timeline(orig_raw, total_samples, offset_ms=0.0, gain_db=orig_gain_db, fade_in_ms=5.0, fade_out_ms=20.0)
    stem_vo = place_stem_on_timeline(vo_raw, total_samples, offset_ms=vo_offset_ms, gain_db=vo_gain_db, fade_in_ms=10.0, fade_out_ms=30.0)
    stem_foley = place_stem_on_timeline(foley_raw, total_samples, offset_ms=foley_offset_ms, gain_db=foley_gain_db, fade_in_ms=5.0, fade_out_ms=40.0)
    stem_bgm = place_stem_on_timeline(bgm_raw, total_samples, offset_ms=0.0, gain_db=bgm_gain_db, loop_to_fill=True, fade_in_ms=150.0, fade_out_ms=600.0)

    speech_bus = stem_orig + stem_vo
    if auto_duck_bgm and bgm_raw is not None and np.max(np.abs(speech_bus)) > 1e-4:
        duck_gain, duck_db_curve = compute_speech_ducking_envelope(
            speech_bus,
            sr=TARGET_SR,
            threshold_db=duck_threshold_db,
            duck_depth_db=duck_depth_db,
            attack_ms=duck_attack_ms,
            hold_ms=duck_hold_ms,
            release_ms=duck_release_ms,
        )
        stem_bgm_ducked = stem_bgm * duck_gain
    else:
        duck_db_curve = np.zeros(total_samples, dtype=np.float32)
        stem_bgm_ducked = stem_bgm

    mix = speech_bus + stem_foley + stem_bgm_ducked

    # Sub-rumble 35 Hz Butterworth high-pass filter on master bus
    if len(mix) > 256:
        try:
            sos_master = sps.butter(2, 35.0, btype="highpass", fs=TARGET_SR, output="sos")
            mix = sps.sosfiltfilt(sos_master, mix, axis=0).astype(np.float32)
        except Exception:
            pass

    # ITU-R BS.1770-4 Loudness Normalization
    measured_lufs = -24.0
    if np.max(np.abs(mix)) > 1e-5:
        try:
            meter = pyln.Meter(TARGET_SR)
            measured_lufs = float(meter.integrated_loudness(mix))
            if np.isfinite(measured_lufs) and measured_lufs > -70.0:
                mix = pyln.normalize.loudness(mix, measured_lufs, float(master_lufs)).astype(np.float32)
                measured_lufs = float(master_lufs)
        except Exception as e:
            print(f"[LUFS] Normalization fallback: {e}")

    # True-peak ceiling limiter
    ceiling = float(10.0 ** (true_peak_db / 20.0))
    peak = float(np.max(np.abs(mix)))
    if peak > ceiling and peak > 1e-6:
        mix = np.tanh(mix / ceiling) * ceiling

    ts = time.strftime("%Y%m%d_%H%M%S")
    stem_slug = Path(video_path).stem[:32]
    out_wav = os.path.join(output_dir, f"{stem_slug}_master_{ts}.wav")
    out_mp4 = os.path.join(output_dir, f"{stem_slug}_scored_{ts}.mp4")
    out_plot = os.path.join(output_dir, f"{stem_slug}_envelope_{ts}.png")

    sf.write(out_wav, mix, TARGET_SR, subtype="PCM_16")

    # Mux mastered stereo WAV into video (stream-copy video track for zero quality loss)
    mux_cmd = [
        "ffmpeg", "-y", "-v", "error",
        "-i", video_path, "-i", out_wav,
        "-map", "0:v:0", "-map", "1:a:0",
        "-c:v", "copy", "-c:a", "aac", "-b:a", "256k",
        "-shortest", out_mp4
    ]
    r = subprocess.run(mux_cmd, capture_output=True, text=True, check=False)
    if r.returncode != 0 or not os.path.isfile(out_mp4):
        # Fallback re-encode video if container rejected stream copy
        mux_cmd_fallback = [
            "ffmpeg", "-y", "-v", "error",
            "-i", video_path, "-i", out_wav,
            "-map", "0:v:0", "-map", "1:a:0",
            "-c:v", "libx264", "-preset", "fast", "-crf", "18", "-pix_fmt", "yuv420p",
            "-c:a", "aac", "-b:a", "256k", "-shortest", out_mp4
        ]
        subprocess.run(mux_cmd_fallback, capture_output=True, text=True, check=True)

    # Render diagnostic plot
    try:
        step = max(1, total_samples // 2500)
        t_axis = np.arange(0, total_samples, step) / float(TARGET_SR)
        fig, axes = plt.subplots(3, 1, figsize=(10.5, 5.2), sharex=True, facecolor="#0b0f19")
        for ax in axes:
            ax.set_facecolor("#111827")
            ax.tick_params(colors="#9ca3af", labelsize=8)
            for spine in ax.spines.values():
                spine.set_color("#1f2937")

        axes[0].plot(t_axis, np.mean(speech_bus[::step], axis=1), color="#38bdf8", lw=0.8, label="Dialogue / Speech Bus")
        if foley_raw is not None:
            axes[0].plot(t_axis, np.mean(stem_foley[::step], axis=1), color="#f59e0b", lw=0.7, alpha=0.8, label="Foley / SFX Stem")
        axes[0].set_ylabel("Stems", color="#e5e7eb", fontsize=9)
        axes[0].legend(loc="upper right", fontsize=8, facecolor="#1f2937", edgecolor="#374151", labelcolor="#f3f4f6")

        axes[1].plot(t_axis, duck_db_curve[::step], color="#f43f5e", lw=1.4, label="BGM Auto-Ducking Gain Reduction (dB)")
        axes[1].fill_between(t_axis, duck_db_curve[::step], 0.0, color="#f43f5e", alpha=0.2)
        axes[1].set_ylim(min(-22.0, float(duck_depth_db) - 3.0), 2.0)
        axes[1].set_ylabel("Duck (dB)", color="#e5e7eb", fontsize=9)
        axes[1].legend(loc="lower right", fontsize=8, facecolor="#1f2937", edgecolor="#374151", labelcolor="#f3f4f6")

        axes[2].plot(t_axis, mix[::step, 0], color="#10b981", lw=0.8, alpha=0.85, label=f"Mastered Bus ({master_lufs:.1f} LUFS / {true_peak_db:.1f} dBTP)")
        axes[2].set_ylabel("Master", color="#e5e7eb", fontsize=9)
        axes[2].set_xlabel("Time (seconds)", color="#e5e7eb", fontsize=9)
        axes[2].legend(loc="upper right", fontsize=8, facecolor="#1f2937", edgecolor="#374151", labelcolor="#f3f4f6")

        plt.tight_layout()
        plt.savefig(out_plot, dpi=140, facecolor="#0b0f19")
        plt.close(fig)
    except Exception as e:
        print(f"[Plot] Diagnostic plot warning: {e}")
        out_plot = None

    return {
        "video_path": out_mp4,
        "wav_path": out_wav,
        "plot_path": out_plot,
        "duration_sec": round(dur_s, 2),
        "measured_lufs": round(measured_lufs, 2),
    }

# ── 4. Word-Level ASR & Styled ASS / SRT Karaoke Subtitle Studio ─────────────
CAPTION_STYLES = {
    "viral_karaoke": {
        "name": "Viral Reels / Shorts Karaoke (Active Word Cyan/Yellow Highlight)",
        "font": "DejaVu Sans",
        "fontsize": 56,
        "primary_ass": "&H0000FFFF",     # Active/highlighted word (Yellow in BGR: 00FFFF)
        "secondary_ass": "&H00FFFFFF",   # Upcoming word (White)
        "outline_ass": "&H00000000",     # Black outline
        "back_ass": "&H80000000",        # Drop shadow
        "border_style": 1,
        "outline": 4,
        "shadow": 2,
        "alignment": 2,                  # Bottom center
        "margin_v": 95,
        "karaoke": True,
    },
    "cinema_classic": {
        "name": "Cinema Classic (Warm White + Crisp Black Outline)",
        "font": "Liberation Sans",
        "fontsize": 46,
        "primary_ass": "&H00F8F8F8",
        "secondary_ass": "&H00F8F8F8",
        "outline_ass": "&H00000000",
        "back_ass": "&H80000000",
        "border_style": 1,
        "outline": 3,
        "shadow": 2,
        "alignment": 2,
        "margin_v": 65,
        "karaoke": False,
    },
    "broadcast_yellow": {
        "name": "Documentary Broadcast Yellow (Bold Yellow + Shadow)",
        "font": "DejaVu Sans",
        "fontsize": 48,
        "primary_ass": "&H0000E5FF",
        "secondary_ass": "&H00FFFFFF",
        "outline_ass": "&H00000000",
        "back_ass": "&H90000000",
        "border_style": 1,
        "outline": 3,
        "shadow": 2,
        "alignment": 2,
        "margin_v": 70,
        "karaoke": False,
    },
    "minimal_pill": {
        "name": "Modern Translucent Pill Box (Opaque Rounded Backdrop)",
        "font": "DejaVu Sans",
        "fontsize": 44,
        "primary_ass": "&H00FFFFFF",
        "secondary_ass": "&H00FFFFFF",
        "outline_ass": "&H60111827",
        "back_ass": "&H60111827",
        "border_style": 3,               # Opaque box in ASS
        "outline": 12,
        "shadow": 0,
        "alignment": 2,
        "margin_v": 75,
        "karaoke": False,
    },
    "cyberpunk_neon": {
        "name": "Cyberpunk Neon HUD (Electric Cyan + Magenta Outline)",
        "font": "DejaVu Sans",
        "fontsize": 50,
        "primary_ass": "&H00FFFF00",     # Cyan (BGR: FFFF00)
        "secondary_ass": "&H00FFFFFF",
        "outline_ass": "&H009900FF",     # Magenta/Violet outline
        "back_ass": "&H80000000",
        "border_style": 1,
        "outline": 4,
        "shadow": 3,
        "alignment": 2,
        "margin_v": 80,
        "karaoke": True,
    },
    "top_banner": {
        "name": "Top Presenter Banner (Upper Third Placement)",
        "font": "DejaVu Sans",
        "fontsize": 44,
        "primary_ass": "&H00FFFFFF",
        "secondary_ass": "&H0000FFFF",
        "outline_ass": "&H00000000",
        "back_ass": "&H80000000",
        "border_style": 1,
        "outline": 3,
        "shadow": 2,
        "alignment": 8,                  # Top center
        "margin_v": 55,
        "karaoke": False,
    },
}

def _fmt_srt_time(seconds: float) -> str:
    seconds = max(0.0, float(seconds))
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    ms = int(round((seconds - int(seconds)) * 1000))
    if ms >= 1000:
        ms = 999
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def _fmt_ass_time(seconds: float) -> str:
    seconds = max(0.0, float(seconds))
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    cs = int(round((seconds - int(seconds)) * 100))
    if cs >= 100:
        cs = 99
    return f"{h:d}:{m:02d}:{s:02d}.{cs:02d}"

def transcribe_media_word_level(
    media_path: str,
    model_id: str = WHISPER_MODEL_SIZE,
    language: str = "auto",
    max_words_per_line: int = 7,
) -> list[dict]:
    """
    Transcribes audio/video into short, readable caption segments with word-level timings:
    Returns list of dicts: [{"start": float, "end": float, "text": str, "words": [{"word": str, "start": float, "end": float}]}]
    """
    if not media_path or not os.path.isfile(media_path):
        raise ValueError("Please provide a valid video or audio file to transcribe.")

    model, backend = load_asr_engine(model_id)
    lang_arg = None if (not language or language == "auto") else language

    raw_words = []
    if backend == "stable_faster_whisper":
        res = model.transcribe_stable(media_path, language=lang_arg, vad=True, word_timestamps=True)
        for seg in res.segments:
            for w in getattr(seg, "words", []) or []:
                w_txt = str(getattr(w, "word", "")).strip()
                if w_txt:
                    raw_words.append({
                        "word": w_txt,
                        "start": round(float(w.start), 3),
                        "end": round(max(float(w.start) + 0.04, float(w.end)), 3),
                    })
    else:
        segments, _ = model.transcribe(media_path, language=lang_arg, vad_filter=True, word_timestamps=True)
        for seg in segments:
            for w in getattr(seg, "words", []) or []:
                w_txt = str(getattr(w, "word", "")).strip()
                if w_txt:
                    raw_words.append({
                        "word": w_txt,
                        "start": round(float(w.start), 3),
                        "end": round(max(float(w.start) + 0.04, float(w.end)), 3),
                    })

    # Regroup words into punchy caption lines (<= max_words_per_line or > 0.65s pause)
    grouped = []
    cur = []
    for w in raw_words:
        if cur:
            gap = w["start"] - cur[-1]["end"]
            punct_break = cur[-1]["word"].endswith((".", "!", "?", ";", ":"))
            if len(cur) >= max_words_per_line or gap > 0.65 or (len(cur) >= 4 and punct_break):
                grouped.append({
                    "start": cur[0]["start"],
                    "end": cur[-1]["end"],
                    "text": " ".join(x["word"] for x in cur),
                    "words": cur,
                })
                cur = []
        cur.append(w)
    if cur:
        grouped.append({
            "start": cur[0]["start"],
            "end": cur[-1]["end"],
            "text": " ".join(x["word"] for x in cur),
            "words": cur,
        })
    return grouped

def export_subtitles_srt_and_ass(
    segments: list[dict],
    out_srt_path: str,
    out_ass_path: str,
    style_key: str = "viral_karaoke",
    play_res_x: int = 1920,
    play_res_y: int = 1080,
    font_scale: float = 1.0,
) -> tuple[str, str]:
    """Writes both standard .srt and styled/karaoke .ass subtitle files."""
    st = CAPTION_STYLES.get(style_key, CAPTION_STYLES["viral_karaoke"])
    scaled_fs = max(20, int(round(st["fontsize"] * font_scale * (play_res_y / 1080.0))))
    margin_v = max(20, int(round(st["margin_v"] * (play_res_y / 1080.0))))

    srt_lines = []
    for idx, seg in enumerate(segments, 1):
        srt_lines.append(f"{idx}")
        srt_lines.append(f"{_fmt_srt_time(seg['start'])} --> {_fmt_srt_time(seg['end'])}")
        srt_lines.append(seg["text"].strip())
        srt_lines.append("")
    with open(out_srt_path, "w", encoding="utf-8") as f:
        f.write("\n".join(srt_lines))

    ass_header = f"""[Script Info]
Title: AEI Director Studio Subtitles
ScriptType: v4.00+
PlayResX: {play_res_x}
PlayResY: {play_res_y}
WrapStyle: 0
ScaledBorderAndShadow: yes

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Default,{st['font']},{scaled_fs},{st['primary_ass']},{st['secondary_ass']},{st['outline_ass']},{st['back_ass']},-1,0,0,0,100,100,0,0,{st['border_style']},{st['outline']},{st['shadow']},{st['alignment']},60,60,{margin_v},1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
"""
    ass_events = []
    for seg in segments:
        s_str = _fmt_ass_time(seg["start"])
        e_str = _fmt_ass_time(seg["end"])
        words = seg.get("words") or []
        if st.get("karaoke") and words:
            k_parts = []
            prev_end = seg["start"]
            for w in words:
                gap_cs = max(0, int(round((w["start"] - prev_end) * 100)))
                if gap_cs > 0:
                    k_parts.append(f"{{\\k{gap_cs}}}")
                dur_cs = max(4, int(round((w["end"] - w["start"]) * 100)))
                clean_w = str(w["word"]).replace("{", "").replace("}", "")
                k_parts.append(f"{{\\kf{dur_cs}}}{clean_w}")
                prev_end = w["end"]
            line_txt = " ".join(k_parts)
        else:
            line_txt = str(seg["text"]).replace("{", "").replace("}", "")
        ass_events.append(f"Dialogue: 0,{s_str},{e_str},Default,,0,0,0,,{line_txt}")

    with open(out_ass_path, "w", encoding="utf-8") as f:
        f.write(ass_header + "\n".join(ass_events) + "\n")
    return out_srt_path, out_ass_path

def burn_ass_subtitles_into_video(video_path: str, ass_path: str, out_video_path: str) -> str:
    """Burns an .ass subtitle file into a video using ffmpeg's libass filter."""
    # Copy .ass to a clean temporary path without colons/spaces so ffmpeg filtergraph parser never chokes
    tmp_ass = os.path.join(tempfile.gettempdir(), f"aei_sub_{int(time.time()*1000)}.ass")
    shutil.copy2(ass_path, tmp_ass)
    escaped_ass = tmp_ass.replace("\\", "/").replace(":", "\\:")
    try:
        cmd = [
            "ffmpeg", "-y", "-v", "error",
            "-i", video_path,
            "-vf", f"ass='{escaped_ass}'",
            "-c:v", "libx264", "-preset", "fast", "-crf", "18", "-pix_fmt", "yuv420p",
            "-c:a", "copy", out_video_path
        ]
        r = subprocess.run(cmd, capture_output=True, text=True, check=False)
        if r.returncode != 0:
            raise RuntimeError(f"ffmpeg libass burn failed: {r.stderr[-600:]}")
        return out_video_path
    finally:
        if os.path.isfile(tmp_ass):
            try:
                os.remove(tmp_ass)
            except OSError:
                pass

# ── 5. Multi-Clip Timeline Sequencer, xfade Transitions & Color Grading ──────
COLOR_GRADE_FILTERS = {
    "neutral": "",
    "teal_orange": "colorbalance=rs=0.06:gs=-0.02:bs=-0.08:rm=0.04:gm=0.0:bm=-0.04:rh=0.08:gh=0.02:bh=-0.06,eq=contrast=1.06:saturation=1.12",
    "golden_hour": "colorbalance=rs=0.08:gs=0.04:bs=-0.06:rm=0.07:gm=0.03:bm=-0.05,eq=contrast=1.04:saturation=1.15:brightness=0.02",
    "kodak_film": "eq=contrast=1.14:saturation=0.92:gamma=0.96,colorbalance=rs=0.03:gs=0.01:bs=-0.03",
    "cyberpunk_neon": "colorbalance=rs=-0.05:gs=-0.04:bs=0.12:rh=0.10:gh=-0.05:bh=0.12,eq=contrast=1.12:saturation=1.28",
    "noir_bw": "hue=s=0,eq=contrast=1.22:brightness=-0.02:gamma=0.94",
}

CANVAS_PRESETS = {
    "16:9_1080p": (1920, 1080),
    "9:16_1080p_vertical": (1080, 1920),
    "1:1_1080p_square": (1080, 1080),
    "16:9_720p": (1280, 720),
    "16:9_4k": (3840, 2160),
}

def normalize_clip_for_timeline(
    clip_path: str,
    out_path: str,
    width: int = 1920,
    height: int = 1080,
    fps: int = 30,
    fit_mode: str = "blurred_backdrop",
    color_grade: str = "neutral",
) -> dict:
    """Normalizes any input video clip to exact (width, height, fps, 48 kHz stereo audio) for seamless xfade stitching."""
    info = probe_video_info(clip_path)
    dur = max(0.5, info["duration"])
    grade_flt = COLOR_GRADE_FILTERS.get(color_grade, "")
    grade_suffix = f",{grade_flt}" if grade_flt else ""

    if fit_mode == "center_crop":
        vf = (
            f"scale={width}:{height}:force_original_aspect_ratio=increase,"
            f"crop={width}:{height},setsar=1,fps={fps}{grade_suffix},format=yuv420p"
        )
    elif fit_mode == "letterbox_black":
        vf = (
            f"scale={width}:{height}:force_original_aspect_ratio=decrease,"
            f"pad={width}:{height}:(ow-iw)/2:(oh-ih)/2:black,setsar=1,fps={fps}{grade_suffix},format=yuv420p"
        )
    else:
        # Blurred glass backdrop for mixed aspect ratios (e.g. 16:9 into 9:16 or vice versa)
        vf = (
            f"split=2[bg][fg];"
            f"[bg]scale={width}:{height}:force_original_aspect_ratio=increase,"
            f"crop={width}:{height},boxblur=24:4,eq=brightness=-0.12[bg_b];"
            f"[fg]scale={width}:{height}:force_original_aspect_ratio=decrease[fg_s];"
            f"[bg_b][fg_s]overlay=(W-w)/2:(H-h)/2,setsar=1,fps={fps}{grade_suffix},format=yuv420p"
        )

    if info["has_audio"]:
        cmd = [
            "ffmpeg", "-y", "-v", "error", "-i", clip_path,
            "-vf", vf,
            "-c:v", "libx264", "-preset", "fast", "-crf", "18",
            "-ac", "2", "-ar", str(TARGET_SR), "-c:a", "aac", "-b:a", "192k",
            out_path
        ]
    else:
        cmd = [
            "ffmpeg", "-y", "-v", "error",
            "-i", clip_path,
            "-f", "lavfi", "-t", f"{dur:.3f}", "-i", f"anullsrc=channel_layout=stereo:sample_rate={TARGET_SR}",
            "-vf", vf,
            "-map", "0:v:0", "-map", "1:a:0",
            "-c:v", "libx264", "-preset", "fast", "-crf", "18",
            "-c:a", "aac", "-b:a", "192k", "-shortest",
            out_path
        ]
    r = subprocess.run(cmd, capture_output=True, text=True, check=False)
    if r.returncode != 0:
        raise RuntimeError(f"Clip normalization failed for {os.path.basename(clip_path)}: {r.stderr[-500:]}")
    return probe_video_info(out_path)

def assemble_timeline_sequence(
    clip_paths: list[str],
    canvas_key: str = "16:9_1080p",
    fps: int = 30,
    fit_mode: str = "blurred_backdrop",
    transition_type: str = "fade",
    transition_dur_sec: float = 0.5,
    color_grade: str = "neutral",
    bgm_path: str | None = None,
    bgm_gain_db: float = -10.0,
    auto_duck_bgm: bool = True,
    master_lufs: float = -14.0,
    output_dir: str = OUTPUT_BASE_DIR,
) -> dict:
    """
    Normalizes 1..N video clips, joins them with ffmpeg xfade/acrossfade (or hard cuts),
    and optionally scores the assembled timeline with an auto-ducked BGM track.
    """
    valid_clips = [p for p in clip_paths if p and os.path.isfile(p)]
    if not valid_clips:
        raise ValueError("Please provide at least 1 valid video clip for timeline assembly.")

    os.makedirs(output_dir, exist_ok=True)
    width, height = CANVAS_PRESETS.get(canvas_key, (1920, 1080))
    tmpdir = tempfile.mkdtemp(prefix="aei_timeline_")

    try:
        norm_paths = []
        norm_durs = []
        for idx, cp in enumerate(valid_clips):
            np_out = os.path.join(tmpdir, f"norm_{idx:03d}.mp4")
            pinfo = normalize_clip_for_timeline(cp, np_out, width=width, height=height, fps=fps, fit_mode=fit_mode, color_grade=color_grade)
            norm_paths.append(np_out)
            norm_durs.append(max(0.5, pinfo["duration"]))

        ts = time.strftime("%Y%m%d_%H%M%S")
        joined_mp4 = os.path.join(tmpdir, f"joined_{ts}.mp4")

        if len(norm_paths) == 1 or transition_type == "cut" or transition_dur_sec <= 0.02:
            concat_txt = os.path.join(tmpdir, "concat.txt")
            with open(concat_txt, "w", encoding="utf-8") as f:
                for p in norm_paths:
                    safe_p = p.replace("'", "'\\''")
                    f.write(f"file '{safe_p}'\n")
            cmd = [
                "ffmpeg", "-y", "-v", "error", "-f", "concat", "-safe", "0",
                "-i", concat_txt, "-c", "copy", joined_mp4
            ]
            subprocess.run(cmd, capture_output=True, text=True, check=True)
        else:
            # Iterative pairwise xfade + acrossfade so even 10+ clips never hit complex filtergraph limits
            cur_video = norm_paths[0]
            cur_dur = norm_durs[0]
            for idx in range(1, len(norm_paths)):
                nxt_video = norm_paths[idx]
                nxt_dur = norm_durs[idx]
                td = min(float(transition_dur_sec), cur_dur * 0.4, nxt_dur * 0.4)
                offset = max(0.05, cur_dur - td)
                step_out = os.path.join(tmpdir, f"xfade_step_{idx:03d}.mp4")
                filter_complex = (
                    f"[0:v][1:v]xfade=transition={transition_type}:duration={td:.3f}:offset={offset:.3f}[v];"
                    f"[0:a][1:a]acrossfade=d={td:.3f}[a]"
                )
                cmd = [
                    "ffmpeg", "-y", "-v", "error",
                    "-i", cur_video, "-i", nxt_video,
                    "-filter_complex", filter_complex,
                    "-map", "[v]", "-map", "[a]",
                    "-c:v", "libx264", "-preset", "fast", "-crf", "18", "-pix_fmt", "yuv420p",
                    "-c:a", "aac", "-b:a", "192k", step_out
                ]
                r = subprocess.run(cmd, capture_output=True, text=True, check=False)
                if r.returncode != 0:
                    raise RuntimeError(f"xfade step {idx} failed: {r.stderr[-500:]}")
                cur_video = step_out
                cur_dur = offset + nxt_dur
            shutil.copy2(cur_video, joined_mp4)

        # Run final audio mastering + optional global BGM auto-ducking across the joined sequence
        scored = mix_and_master_scene(
            video_path=joined_mp4,
            bgm_path=bgm_path,
            bgm_gain_db=bgm_gain_db,
            auto_duck_bgm=auto_duck_bgm,
            master_lufs=master_lufs,
            output_dir=output_dir,
        )
        final_mp4 = os.path.join(output_dir, f"timeline_master_{canvas_key}_{ts}.mp4")
        shutil.move(scored["video_path"], final_mp4)
        scored["video_path"] = final_mp4
        scored["num_clips"] = len(valid_clips)
        return scored
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)

print("[OK] Step 3 Core Post-Production & Mastering Engine ready.")


In [ ]:
# @title STEP 4 — Launch AEI Director Studio Gradio UI
# @markdown Launches the 4-tab Gradio 5.49.1 Studio:
# @markdown 1. **🎬 Single-Scene 4-Stem Scorer & Auto-Ducking Mixer**
# @markdown 2. **💬 AI Word-Level Subtitle & Karaoke Caption Studio (`faster-whisper` + `stable-ts`)**
# @markdown 3. **🎞️ Multi-Clip Timeline Sequencer, `xfade` Transitions & Aspect-Ratio Finisher**
# @markdown 4. **📦 JSON Timeline Manifest & Batch Mastering Queue**

from IPython.display import clear_output
clear_output(wait=True)

import os, json, time, traceback
import gradio as gr

LAST_SUBTITLE_SEGMENTS = []

def _refresh_drive_choices():
    global DRIVE_ASSETS
    DRIVE_ASSETS = scan_drive_media_assets()
    return (
        gr.Dropdown(choices=[""] + DRIVE_ASSETS["videos"], value=""),
        gr.Dropdown(choices=[""] + DRIVE_ASSETS["dialogues"], value=""),
        gr.Dropdown(choices=[""] + DRIVE_ASSETS["foleys"], value=""),
        gr.Dropdown(choices=[""] + DRIVE_ASSETS["bgms"], value=""),
        gr.Dropdown(choices=DRIVE_ASSETS["videos"], value=[]),
        f"✅ Indexed **{len(DRIVE_ASSETS['videos'])}** videos, **{len(DRIVE_ASSETS['dialogues'])}** dialogue/TTS clips, **{len(DRIVE_ASSETS['foleys'])}** foley/SFX clips, and **{len(DRIVE_ASSETS['bgms'])}** BGM tracks.",
    )

def ui_score_single_scene(
    upload_video, drive_video,
    upload_vo, drive_vo,
    upload_foley, drive_foley,
    upload_bgm, drive_bgm,
    orig_gain_db, mute_orig,
    vo_gain_db, vo_offset_ms,
    foley_gain_db, foley_offset_ms,
    bgm_gain_db, auto_duck, duck_thresh_db, duck_depth_db, duck_attack_ms, duck_hold_ms, duck_release_ms,
    master_lufs, true_peak_db,
):
    try:
        v_path = upload_video or (drive_video if drive_video else None)
        vo_path = upload_vo or (drive_vo if drive_vo else None)
        fol_path = upload_foley or (drive_foley if drive_foley else None)
        bgm_path = upload_bgm or (drive_bgm if drive_bgm else None)

        res = mix_and_master_scene(
            video_path=v_path,
            vo_path=vo_path,
            foley_path=fol_path,
            bgm_path=bgm_path,
            orig_gain_db=float(orig_gain_db),
            mute_orig_audio=bool(mute_orig),
            vo_gain_db=float(vo_gain_db),
            vo_offset_ms=float(vo_offset_ms),
            foley_gain_db=float(foley_gain_db),
            foley_offset_ms=float(foley_offset_ms),
            bgm_gain_db=float(bgm_gain_db),
            auto_duck_bgm=bool(auto_duck),
            duck_threshold_db=float(duck_thresh_db),
            duck_depth_db=float(duck_depth_db),
            duck_attack_ms=float(duck_attack_ms),
            duck_hold_ms=float(duck_hold_ms),
            duck_release_ms=float(duck_release_ms),
            master_lufs=float(master_lufs),
            true_peak_db=float(true_peak_db),
        )
        msg = (
            f"✅ **Mastered Scene Ready!**\n"
            f"- **Video Output**: `{res['video_path']}` (`{res['duration_sec']}s`)\n"
            f"- **Master Stereo WAV**: `{res['wav_path']}` (`{res['measured_lufs']} LUFS`)"
        )
        return res["video_path"], res["wav_path"], res["plot_path"], msg
    except Exception as e:
        traceback.print_exc()
        return None, None, None, f"❌ **Error**: {e}"

def ui_transcribe_video(upload_video, drive_video, model_id, language, max_words_line):
    global LAST_SUBTITLE_SEGMENTS
    try:
        v_path = upload_video or (drive_video if drive_video else None)
        segs = transcribe_media_word_level(
            media_path=v_path,
            model_id=model_id,
            language=language,
            max_words_per_line=int(max_words_line),
        )
        LAST_SUBTITLE_SEGMENTS = segs
        rows = [[round(s["start"], 2), round(s["end"], 2), s["text"]] for s in segs]
        return rows, f"✅ Transcribed **{len(segs)}** subtitle lines with word-level timestamps. Edit any row below and click **Burn & Export Subtitles**."
    except Exception as e:
        traceback.print_exc()
        return [], f"❌ **Transcription Error**: {e}"

def ui_burn_subtitles(upload_video, drive_video, table_rows, style_key, font_scale):
    global LAST_SUBTITLE_SEGMENTS
    try:
        v_path = upload_video or (drive_video if drive_video else None)
        if not v_path or not os.path.isfile(v_path):
            raise ValueError("Please upload or select a video file first.")

        # Sync user edits from table_rows back into LAST_SUBTITLE_SEGMENTS
        segs = []
        if hasattr(table_rows, "values"):
            table_rows = table_rows.values.tolist()
        for idx, row in enumerate(table_rows or []):
            if len(row) < 3 or not str(row[2]).strip():
                continue
            st_t = float(row[0])
            en_t = max(st_t + 0.1, float(row[1]))
            txt = str(row[2]).strip()
            orig_words = LAST_SUBTITLE_SEGMENTS[idx].get("words", []) if idx < len(LAST_SUBTITLE_SEGMENTS) else []
            orig_txt = LAST_SUBTITLE_SEGMENTS[idx].get("text", "").strip() if idx < len(LAST_SUBTITLE_SEGMENTS) else ""
            if txt != orig_txt or not orig_words:
                # Redistribute word timings evenly across edited text
                tokens = txt.split()
                n_tok = max(1, len(tokens))
                step_t = (en_t - st_t) / n_tok
                orig_words = [
                    {"word": tok, "start": round(st_t + i * step_t, 3), "end": round(st_t + (i + 1) * step_t, 3)}
                    for i, tok in enumerate(tokens)
                ]
            segs.append({"start": st_t, "end": en_t, "text": txt, "words": orig_words})

        if not segs:
            raise ValueError("No subtitle segments found. Run '1. Transcribe Word-Level Captions' first or add rows to the table.")

        vinfo = probe_video_info(v_path)
        ts = time.strftime("%Y%m%d_%H%M%S")
        slug = Path(v_path).stem[:32]
        out_srt = os.path.join(OUTPUT_BASE_DIR, f"{slug}_{ts}.srt")
        out_ass = os.path.join(OUTPUT_BASE_DIR, f"{slug}_{ts}.ass")
        out_mp4 = os.path.join(OUTPUT_BASE_DIR, f"{slug}_subtitled_{ts}.mp4")

        export_subtitles_srt_and_ass(
            segs, out_srt, out_ass,
            style_key=style_key,
            play_res_x=vinfo["width"],
            play_res_y=vinfo["height"],
            font_scale=float(font_scale),
        )
        burn_ass_subtitles_into_video(v_path, out_ass, out_mp4)
        return (
            out_mp4,
            out_srt,
            out_ass,
            f"✅ **Subtitled Video & Caption Files Exported!**\n- **Burned MP4**: `{out_mp4}`\n- **Karaoke ASS**: `{out_ass}`\n- **SRT**: `{out_srt}`",
        )
    except Exception as e:
        traceback.print_exc()
        return None, None, None, f"❌ **Subtitle Burn Error**: {e}"

def ui_assemble_timeline(
    uploaded_files, drive_selected_videos,
    canvas_key, fps, fit_mode, transition_type, transition_dur, color_grade,
    upload_bgm, drive_bgm, bgm_gain_db, auto_duck, master_lufs,
):
    try:
        clips = []
        if uploaded_files:
            for f in uploaded_files:
                clips.append(f if isinstance(f, str) else getattr(f, "name", str(f)))
        if drive_selected_videos:
            clips.extend([p for p in drive_selected_videos if p])
        bgm_path = upload_bgm or (drive_bgm if drive_bgm else None)

        res = assemble_timeline_sequence(
            clip_paths=clips,
            canvas_key=canvas_key,
            fps=int(fps),
            fit_mode=fit_mode,
            transition_type=transition_type,
            transition_dur_sec=float(transition_dur),
            color_grade=color_grade,
            bgm_path=bgm_path,
            bgm_gain_db=float(bgm_gain_db),
            auto_duck_bgm=bool(auto_duck),
            master_lufs=float(master_lufs),
        )
        msg = (
            f"✅ **Assembled {res['num_clips']} Clips into Mastered Timeline!**\n"
            f"- **Output MP4**: `{res['video_path']}` (`{res['duration_sec']}s`, `{canvas_key}`, `{fps} fps`)\n"
            f"- **Master Loudness**: `{res['measured_lufs']} LUFS`"
        )
        return res["video_path"], res["plot_path"], msg
    except Exception as e:
        traceback.print_exc()
        return None, None, f"❌ **Timeline Assembly Error**: {e}"

def ui_run_json_manifest(manifest_json_str):
    try:
        spec = json.loads(manifest_json_str)
        res = assemble_timeline_sequence(
            clip_paths=spec.get("clips", []),
            canvas_key=spec.get("canvas", "16:9_1080p"),
            fps=int(spec.get("fps", 30)),
            fit_mode=spec.get("fit_mode", "blurred_backdrop"),
            transition_type=spec.get("transition", "fade"),
            transition_dur_sec=float(spec.get("transition_dur", 0.5)),
            color_grade=spec.get("color_grade", "neutral"),
            bgm_path=spec.get("bgm_path"),
            bgm_gain_db=float(spec.get("bgm_gain_db", -10.0)),
            auto_duck_bgm=bool(spec.get("auto_duck_bgm", True)),
            master_lufs=float(spec.get("master_lufs", -14.0)),
        )
        return res["video_path"], f"✅ **Manifest Rendered**: `{res['video_path']}` (`{res['duration_sec']}s`)"
    except Exception as e:
        traceback.print_exc()
        return None, f"❌ **JSON Manifest Error**: {e}"

with gr.Blocks(title="AEI Director Studio — Multi-Modal Post-Production", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# 🎬 AEI Director Studio — Multi-Modal Timeline Assembler, Auto-Ducking Mixer & Karaoke Subtitler\n"
        "Combine video clips (`Wan2.2`, `FramePack`, `InfiniteTalk`, `SeedVR2`), voiceovers (`Ming-3-TTS`, `Qwen3-TTS`, `VoxCPM2`), "
        "foley (`HunyuanVideo-Foley`, `MMAudio`, `Stable-Audio-3`), and auto-ducked BGM (`ACE-Step`, `MiniMax-Music3`) with word-level `.ass` karaoke captions."
    )

    with gr.Row():
        refresh_btn = gr.Button("🔄 Re-Scan Google Drive Asset Folders (AEI_3D_Out & AEI_TTS_Out)", variant="secondary")
        scan_status_md = gr.Markdown(
            f"📂 Indexed **{len(DRIVE_ASSETS['videos'])}** videos, **{len(DRIVE_ASSETS['dialogues'])}** dialogue clips, "
            f"**{len(DRIVE_ASSETS['foleys'])}** foley/SFX clips, and **{len(DRIVE_ASSETS['bgms'])}** BGM tracks."
        )

    with gr.Tabs():
        # ── TAB 1: Single-Scene 4-Stem Scorer ────────────────────────────────
        with gr.TabItem("🎬 1. Single-Scene 4-Stem Scorer & Auto-Ducking Mixer"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### 1. Video & Audio Stems (Upload or Pick from Drive)")
                    t1_up_vid = gr.Video(label="Upload Primary Video Clip (.mp4/.mov/.webm)", sources=["upload"])
                    t1_dr_vid = gr.Dropdown(
                        choices=[""] + DRIVE_ASSETS["videos"], value="",
                        label="Or Pick Video from Google Drive (AEI_3D_Out)",
                        info="Automatically lists recent videos from Wan2.2, FramePack, InfiniteTalk, LongCat, SeedVR2, etc.",
                    )
                    with gr.Accordion("🗣️ Optional Voiceover / Dialogue Stem (Ming-3-TTS, Qwen3-TTS, VoxCPM2)", open=False):
                        t1_up_vo = gr.Audio(label="Upload Voiceover / TTS Stem", type="filepath")
                        t1_dr_vo = gr.Dropdown(
                            choices=[""] + DRIVE_ASSETS["dialogues"], value="",
                            label="Or Pick Dialogue Stem from Drive",
                            info="Select a TTS / voice-cloning output from AEI_TTS_Out.",
                        )
                        t1_vo_gain = gr.Slider(-24.0, 12.0, value=0.0, step=0.5, label="Voiceover Gain (dB)", info="Relative level adjustment for the voiceover stem.")
                        t1_vo_offset = gr.Slider(-2000.0, 10000.0, value=0.0, step=25.0, label="Voiceover Start Offset (ms)", info="Delay (+) or advance (-) the voiceover stem in milliseconds.")

                    with gr.Accordion("🔊 Optional Synchronized Foley / SFX Stem (HunyuanVideo-Foley, MMAudio, Stable-Audio-3)", open=False):
                        t1_up_fol = gr.Audio(label="Upload Foley / SFX Stem", type="filepath")
                        t1_dr_fol = gr.Dropdown(
                            choices=[""] + DRIVE_ASSETS["foleys"], value="",
                            label="Or Pick Foley / SFX Stem from Drive",
                            info="Select a foley or sound effect stem from your Drive folders.",
                        )
                        t1_fol_gain = gr.Slider(-24.0, 12.0, value=0.0, step=0.5, label="Foley / SFX Gain (dB)", info="Relative level adjustment for the foley stem.")
                        t1_fol_offset = gr.Slider(-2000.0, 2000.0, value=0.0, step=10.0, label="Foley Sync Nudge Offset (ms)", info="Fine-tune transient alignment against visual impacts (-2000 ms to +2000 ms).")

                    with gr.Accordion("🎵 Optional Background Music (BGM) + Sidechain Speech Ducking (ACE-Step, MiniMax-Music3)", open=True):
                        t1_up_bgm = gr.Audio(label="Upload Background Music (BGM)", type="filepath")
                        t1_dr_bgm = gr.Dropdown(
                            choices=[""] + DRIVE_ASSETS["bgms"], value="",
                            label="Or Pick BGM Track from Drive",
                            info="Automatically loops with crossfade if shorter than video, or fades out smoothly at the end.",
                        )
                        t1_bgm_gain = gr.Slider(-30.0, 6.0, value=-8.0, step=0.5, label="BGM Base Gain (dB)", info="Base level of the music bed when nobody is speaking.")
                        t1_autoduck = gr.Checkbox(value=True, label="Enable Sidechain Speech Auto-Ducking", info="Automatically lowers BGM volume whenever speech energy is detected in Video Audio or Voiceover.")
                        with gr.Row():
                            t1_duck_thresh = gr.Slider(-55.0, -15.0, value=-36.0, step=1.0, label="Speech Detection Threshold (dBFS)", info="Vocal-band RMS level that triggers BGM ducking.")
                            t1_duck_depth = gr.Slider(-24.0, -3.0, value=-12.0, step=0.5, label="Ducking Attenuation Depth (dB)", info="How many dB to lower the BGM while speech is active.")
                        with gr.Row():
                            t1_duck_att = gr.Slider(10.0, 200.0, value=45.0, step=5.0, label="Attack (ms)", info="How quickly BGM ducks down when speech starts.")
                            t1_duck_hold = gr.Slider(50.0, 800.0, value=250.0, step=25.0, label="Hold (ms)", info="Prevents pumping between syllables.")
                            t1_duck_rel = gr.Slider(150.0, 1500.0, value=650.0, step=25.0, label="Release (ms)", info="How smoothly BGM swells back up after speech ends.")

                    with gr.Accordion("🎚️ Original Video Audio & Master Bus Loudness (ITU-R BS.1770-4)", open=True):
                        with gr.Row():
                            t1_orig_gain = gr.Slider(-24.0, 12.0, value=0.0, step=0.5, label="Original Video Audio Gain (dB)", info="Adjust volume of the audio already embedded in the video.")
                            t1_mute_orig = gr.Checkbox(value=False, label="Mute Original Video Audio", info="Check to strip the video's original audio track completely.")
                        with gr.Row():
                            t1_lufs = gr.Slider(-23.0, -9.0, value=-14.0, step=0.5, label="Master Target Loudness (LUFS)", info="-14 LUFS for YouTube/Reels/Web; -16 LUFS for Podcast/Apple; -23 LUFS for EBU Broadcast.")
                            t1_peak = gr.Slider(-3.0, -0.1, value=-1.0, step=0.1, label="True-Peak Ceiling (dBTP)", info="Soft-knee limiter ceiling to prevent digital clipping.")

                    t1_btn = gr.Button("🎬 Mix 4 Stems, Auto-Duck BGM & Master Video", variant="primary")

                with gr.Column(scale=1):
                    t1_out_vid = gr.Video(label="Mastered & Scored Video (.mp4)")
                    t1_out_wav = gr.Audio(label="Mastered Stereo Bus (.wav, 48 kHz 16-bit)", type="filepath")
                    t1_out_plot = gr.Image(label="4-Stem Waveform & Sidechain Ducking Envelope Diagnostic", type="filepath")
                    t1_status = gr.Markdown()

        # ── TAB 2: Word-Level Subtitle & Karaoke Caption Studio ──────────────
        with gr.TabItem("💬 2. AI Word-Level Subtitle & Karaoke Caption Studio"):
            with gr.Row():
                with gr.Column(scale=1):
                    t2_up_vid = gr.Video(label="Upload Video to Subtitle", sources=["upload"])
                    t2_dr_vid = gr.Dropdown(
                        choices=[""] + DRIVE_ASSETS["videos"], value="",
                        label="Or Pick Video from Google Drive",
                        info="Select any scored or raw video clip from your Drive.",
                    )
                    with gr.Row():
                        t2_model = gr.Dropdown(
                            choices=["deepdml/faster-whisper-large-v3-turbo-ct2", "large-v3", "medium", "small", "base", "tiny"],
                            value=WHISPER_MODEL_SIZE,
                            label="Whisper ASR Checkpoint",
                            info="large-v3-turbo offers near-large-v3 accuracy at ~4x higher speed.",
                        )
                        t2_lang = gr.Dropdown(
                            choices=["auto", "en", "zh", "ja", "ko", "es", "fr", "de", "pt", "it", "ru", "ar", "hi"],
                            value="auto",
                            label="Spoken Language",
                            info="Set 'auto' for automatic language detection.",
                        )
                        t2_max_words = gr.Slider(3, 12, value=7, step=1, label="Max Words per Caption Line", info="Use 4–5 for vertical Reels/Shorts; 7–9 for 16:9 cinema.")

                    t2_transcribe_btn = gr.Button("🎙️ 1. Transcribe Word-Level Captions (faster-whisper + stable-ts)", variant="secondary")

                    t2_table = gr.Dataframe(
                        headers=["Start (s)", "End (s)", "Caption Text (Editable)"],
                        datatype=["number", "number", "str"],
                        col_count=(3, "fixed"),
                        interactive=True,
                        label="Editable Word-Aligned Subtitle Lines",
                    )

                    with gr.Row():
                        t2_style = gr.Dropdown(
                            choices=[(v["name"], k) for k, v in CAPTION_STYLES.items()],
                            value="viral_karaoke",
                            label="Caption Visual Style Preset (.ass)",
                            info="Viral Karaoke highlights each spoken word in real time via ASS {\\kf} tags.",
                        )
                        t2_font_scale = gr.Slider(0.6, 1.6, value=1.0, step=0.05, label="Font Size Scale", info="Scale subtitle font size relative to video resolution.")

                    t2_burn_btn = gr.Button("🔥 2. Burn Captions into Video & Export .SRT / .ASS", variant="primary")

                with gr.Column(scale=1):
                    t2_out_vid = gr.Video(label="Subtitled Video (.mp4)")
                    with gr.Row():
                        t2_out_srt = gr.File(label="Download .SRT Subtitles")
                        t2_out_ass = gr.File(label="Download .ASS Karaoke Subtitles")
                    t2_status = gr.Markdown()

        # ── TAB 3: Multi-Clip Timeline Sequencer ─────────────────────────────
        with gr.TabItem("🎞️ 3. Multi-Clip Timeline Sequencer, Transitions & Color Finisher"):
            with gr.Row():
                with gr.Column(scale=1):
                    t3_files = gr.File(
                        label="Upload Multiple Video Clips (Ordered 1..N)",
                        file_count="multiple",
                        file_types=["video"],
                    )
                    t3_drive_multi = gr.Dropdown(
                        choices=DRIVE_ASSETS["videos"],
                        value=[],
                        multiselect=True,
                        label="Or Multi-Select Clips from Google Drive (Appended in Selection Order)",
                        info="Pick 2 to 12 clips across Wan2.2, FramePack, InfiniteTalk, LTX-2.5, SeedVR2, etc.",
                    )
                    with gr.Row():
                        t3_canvas = gr.Dropdown(
                            choices=list(CANVAS_PRESETS.keys()),
                            value="16:9_1080p",
                            label="Target Delivery Canvas",
                            info="Normalizes all clips to this resolution.",
                        )
                        t3_fps = gr.Dropdown(
                            choices=[24, 25, 30, 60],
                            value=30,
                            label="Target Frame Rate (FPS)",
                            info="Normalizes frame timing before xfade stitching.",
                        )
                        t3_fit = gr.Dropdown(
                            choices=["blurred_backdrop", "center_crop", "letterbox_black"],
                            value="blurred_backdrop",
                            label="Aspect-Ratio Fit Mode",
                            info="Blurred backdrop fills 9:16 or 16:9 mismatches with a frosted glass background.",
                        )
                    with gr.Row():
                        t3_trans = gr.Dropdown(
                            choices=["fade", "dissolve", "cut", "wipeleft", "wiperight", "slideup", "circleopen", "pixelize"],
                            value="fade",
                            label="Scene Transition Type (ffmpeg xfade)",
                            info="Applies both visual xfade and audio acrossfade between clips.",
                        )
                        t3_trans_dur = gr.Slider(0.1, 1.5, value=0.5, step=0.05, label="Transition Duration (seconds)", info="Overlap duration for each transition.")
                        t3_grade = gr.Dropdown(
                            choices=list(COLOR_GRADE_FILTERS.keys()),
                            value="neutral",
                            label="Cinematic Color Grade Preset",
                            info="Applies consistent color grading across all assembled scenes.",
                        )
                    with gr.Accordion("🎵 Global Timeline BGM + Auto-Ducking", open=False):
                        t3_up_bgm = gr.Audio(label="Upload Global BGM Track", type="filepath")
                        t3_dr_bgm = gr.Dropdown(choices=[""] + DRIVE_ASSETS["bgms"], value="", label="Or Pick BGM from Drive", info="Loops or fades across the entire assembled multi-scene video.")
                        with gr.Row():
                            t3_bgm_gain = gr.Slider(-30.0, 6.0, value=-10.0, step=0.5, label="BGM Gain (dB)", info="Base BGM volume.")
                            t3_autoduck = gr.Checkbox(value=True, label="Auto-Duck BGM Under Scene Dialogue", info="Automatically ducks BGM whenever any scene has active speech.")
                            t3_lufs = gr.Slider(-23.0, -9.0, value=-14.0, step=0.5, label="Master Target Loudness (LUFS)", info="Final ITU-R BS.1770-4 loudness target.")

                    t3_btn = gr.Button("🎞️ Assemble Multi-Clip Timeline & Master Audio", variant="primary")

                with gr.Column(scale=1):
                    t3_out_vid = gr.Video(label="Assembled & Mastered Timeline Video (.mp4)")
                    t3_out_plot = gr.Image(label="Full-Timeline Audio & Ducking Envelope Plot", type="filepath")
                    t3_status = gr.Markdown()

        # ── TAB 4: JSON Project Manifest ─────────────────────────────────────
        with gr.TabItem("📦 4. JSON Project Manifest & Reproducible EDL"):
            gr.Markdown("Define a reproducible multi-scene timeline assembly job in JSON format:")
            default_manifest = json.dumps({
                "clips": DRIVE_ASSETS["videos"][:2] if len(DRIVE_ASSETS["videos"]) >= 2 else ["/content/sample_scene1.mp4", "/content/sample_scene2.mp4"],
                "canvas": "16:9_1080p",
                "fps": 30,
                "fit_mode": "blurred_backdrop",
                "transition": "fade",
                "transition_dur": 0.5,
                "color_grade": "teal_orange",
                "bgm_path": DRIVE_ASSETS["bgms"][0] if DRIVE_ASSETS["bgms"] else None,
                "bgm_gain_db": -10.0,
                "auto_duck_bgm": True,
                "master_lufs": -14.0,
            }, indent=2)
            t4_json = gr.Code(value=default_manifest, language="json", label="Timeline Project Manifest (JSON)")
            t4_btn = gr.Button("🚀 Render JSON Project Manifest", variant="primary")
            t4_out_vid = gr.Video(label="Rendered Project Video (.mp4)")
            t4_status = gr.Markdown()

    refresh_btn.click(
        fn=_refresh_drive_choices,
        inputs=[],
        outputs=[t1_dr_vid, t1_dr_vo, t1_dr_fol, t1_dr_bgm, t3_drive_multi, scan_status_md],
        concurrency_limit=1,
    )
    t1_btn.click(
        fn=ui_score_single_scene,
        inputs=[
            t1_up_vid, t1_dr_vid, t1_up_vo, t1_dr_vo, t1_up_fol, t1_dr_fol, t1_up_bgm, t1_dr_bgm,
            t1_orig_gain, t1_mute_orig, t1_vo_gain, t1_vo_offset, t1_fol_gain, t1_fol_offset,
            t1_bgm_gain, t1_autoduck, t1_duck_thresh, t1_duck_depth, t1_duck_att, t1_duck_hold, t1_duck_rel,
            t1_lufs, t1_peak,
        ],
        outputs=[t1_out_vid, t1_out_wav, t1_out_plot, t1_status],
        concurrency_limit=1,
    )
    t2_transcribe_btn.click(
        fn=ui_transcribe_video,
        inputs=[t2_up_vid, t2_dr_vid, t2_model, t2_lang, t2_max_words],
        outputs=[t2_table, t2_status],
        concurrency_limit=1,
    )
    t2_burn_btn.click(
        fn=ui_burn_subtitles,
        inputs=[t2_up_vid, t2_dr_vid, t2_table, t2_style, t2_font_scale],
        outputs=[t2_out_vid, t2_out_srt, t2_out_ass, t2_status],
        concurrency_limit=1,
    )
    t3_btn.click(
        fn=ui_assemble_timeline,
        inputs=[
            t3_files, t3_drive_multi, t3_canvas, t3_fps, t3_fit, t3_trans, t3_trans_dur, t3_grade,
            t3_up_bgm, t3_dr_bgm, t3_bgm_gain, t3_autoduck, t3_lufs,
        ],
        outputs=[t3_out_vid, t3_out_plot, t3_status],
        concurrency_limit=1,
    )
    t4_btn.click(
        fn=ui_run_json_manifest,
        inputs=[t4_json],
        outputs=[t4_out_vid, t4_status],
        concurrency_limit=1,
    )
    demo.load(
        fn=lambda: f"🎬 **AEI Director Studio Ready.** Output directory: `{OUTPUT_BASE_DIR}`",
        inputs=None,
        outputs=[t1_status],
    )

demo.queue(default_concurrency_limit=1).launch(share=True, inline=True, debug=False, show_error=True)


In [ ]:
# @title STEP 5 — Keep-Alive Cell (Optional)
# @markdown Keeps the Colab session active during long multi-scene rendering jobs.

import time
KEEP_ALIVE_MINUTES = 60  # @param {type:"integer"}
try:
    for m in range(KEEP_ALIVE_MINUTES):
        print(f"[Keep-Alive] Active ({m + 1}/{KEEP_ALIVE_MINUTES} min)...", end="\r", flush=True)
        time.sleep(60)
except KeyboardInterrupt:
    print("\n[Keep-Alive] Stopped by user.")


In [ ]:
# @title STEP 6 — Quick Test (Self-Contained 2-Scene Demo + Auto-Ducking BGM + Karaoke ASS Burn)
# @markdown Synthesizes two test video scenes with simulated speech bursts, a stereo musical chord pad, and a foley ping, then runs **4-Stem Auto-Ducking Mastering**, **Karaoke `.ass` Subtitle Burning**, and **2-Scene `xfade` Assembly** end-to-end.

import os, subprocess
import numpy as np
import soundfile as sf
from IPython.display import display, Video, Audio, Image as IPyImage, FileLink

test_dir = os.path.join(OUTPUT_BASE_DIR, "quick_test")
os.makedirs(test_dir, exist_ok=True)

sr = TARGET_SR
dur = 4.0
t = np.linspace(0.0, dur, int(sr * dur), endpoint=False, dtype=np.float32)

# 1. Create synthetic speech-like bursts (active 0.6s..1.8s and 2.5s..3.4s)
env_speech = ((t >= 0.6) & (t <= 1.8)).astype(np.float32) + ((t >= 2.5) & (t <= 3.4)).astype(np.float32)
speech_sig = 0.35 * env_speech * (np.sin(2 * np.pi * 220 * t) + 0.5 * np.sin(2 * np.pi * 440 * t) + 0.3 * np.sin(2 * np.pi * 880 * t))
vo_wav = os.path.join(test_dir, "demo_vo.wav")
sf.write(vo_wav, np.column_stack([speech_sig, speech_sig]), sr)

# 2. Create continuous stereo BGM pad
bgm_l = 0.25 * (np.sin(2 * np.pi * 261.63 * t) + np.sin(2 * np.pi * 329.63 * t) + np.sin(2 * np.pi * 392.00 * t))
bgm_r = 0.25 * (np.sin(2 * np.pi * 261.63 * t) + np.sin(2 * np.pi * 329.63 * t) + np.sin(2 * np.pi * 523.25 * t))
bgm_wav = os.path.join(test_dir, "demo_bgm.wav")
sf.write(bgm_wav, np.column_stack([bgm_l, bgm_r]), sr)

# 3. Create two 4-second test video clips via ffmpeg testsrc2
clip1 = os.path.join(test_dir, "scene_01.mp4")
clip2 = os.path.join(test_dir, "scene_02.mp4")
subprocess.run([
    "ffmpeg", "-y", "-v", "error", "-f", "lavfi", "-i", "testsrc2=size=1280x720:rate=30:duration=4",
    "-i", vo_wav, "-c:v", "libx264", "-pix_fmt", "yuv420p", "-c:a", "aac", "-shortest", clip1
], check=True)
subprocess.run([
    "ffmpeg", "-y", "-v", "error", "-f", "lavfi", "-i", "testsrc=size=1280x720:rate=30:duration=4",
    "-i", vo_wav, "-c:v", "libx264", "-pix_fmt", "yuv420p", "-c:a", "aac", "-shortest", clip2
], check=True)

# 4. Run 4-stem auto-ducking mixer + karaoke subtitle burn
scored = mix_and_master_scene(
    video_path=clip1,
    bgm_path=bgm_wav,
    bgm_gain_db=-6.0,
    auto_duck_bgm=True,
    duck_depth_db=-14.0,
    master_lufs=-14.0,
    output_dir=test_dir,
)

demo_segs = [
    {
        "start": 0.6, "end": 1.8, "text": "Auto ducking lowers the music",
        "words": [
            {"word": "Auto", "start": 0.6, "end": 0.85},
            {"word": "ducking", "start": 0.85, "end": 1.15},
            {"word": "lowers", "start": 1.15, "end": 1.45},
            {"word": "the", "start": 1.45, "end": 1.6},
            {"word": "music", "start": 1.6, "end": 1.8},
        ],
    },
    {
        "start": 2.5, "end": 3.4, "text": "With karaoke word highlighting!",
        "words": [
            {"word": "With", "start": 2.5, "end": 2.7},
            {"word": "karaoke", "start": 2.7, "end": 3.0},
            {"word": "word", "start": 3.0, "end": 3.18},
            {"word": "highlighting!", "start": 3.18, "end": 3.4},
        ],
    },
]
srt_p, ass_p = export_subtitles_srt_and_ass(
    demo_segs,
    os.path.join(test_dir, "demo.srt"),
    os.path.join(test_dir, "demo.ass"),
    style_key="viral_karaoke",
    play_res_x=1280,
    play_res_y=720,
)
subtitled_mp4 = os.path.join(test_dir, "scene_01_subtitled.mp4")
burn_ass_subtitles_into_video(scored["video_path"], ass_p, subtitled_mp4)

print(f"[Quick Test] Mastered & Subtitled MP4: {subtitled_mp4}")
if scored.get("plot_path") and os.path.isfile(scored["plot_path"]):
    display(IPyImage(filename=scored["plot_path"]))
display(FileLink(subtitled_mp4))
display(FileLink(scored["wav_path"]))


In [ ]:
# @title STEP 7 — Batch Folder Mastering & STEP 8 — Log / Error Inspector
# @markdown Batch-normalizes and masters all `.mp4` videos in a folder to `-14 LUFS` with optional auto-ducked BGM and word-level `.srt`/`.ass` caption generation.

import os, glob, traceback
from IPython.display import display, FileLink

BATCH_INPUT_DIR = "/content/drive/MyDrive/AEI_3D_Out/Director_Studio/quick_test"  # @param {type:"string"}
BATCH_BGM_PATH = ""  # @param {type:"string"}
BATCH_AUTO_SUBTITLE = False  # @param {type:"boolean"}
BATCH_SUBTITLE_STYLE = "cinema_classic"  # @param ["viral_karaoke", "cinema_classic", "broadcast_yellow", "minimal_pill", "cyberpunk_neon", "top_banner"]
BATCH_TARGET_LUFS = -14.0  # @param {type:"number"}

batch_videos = sorted(glob.glob(os.path.join(BATCH_INPUT_DIR, "*.mp4")))
batch_videos = [v for v in batch_videos if "_scored_" not in v and "_subtitled" not in v and "_batch_master" not in v]
print(f"[Batch] Found {len(batch_videos)} raw video(s) in {BATCH_INPUT_DIR}")

for idx, vp in enumerate(batch_videos, 1):
    try:
        print(f"  [{idx}/{len(batch_videos)}] Mastering {os.path.basename(vp)}...")
        res = mix_and_master_scene(
            video_path=vp,
            bgm_path=BATCH_BGM_PATH if (BATCH_BGM_PATH and os.path.isfile(BATCH_BGM_PATH)) else None,
            auto_duck_bgm=True,
            master_lufs=float(BATCH_TARGET_LUFS),
            output_dir=OUTPUT_BASE_DIR,
        )
        final_p = res["video_path"]
        if BATCH_AUTO_SUBTITLE:
            segs = transcribe_media_word_level(final_p)
            if segs:
                vinfo = probe_video_info(final_p)
                srt_p, ass_p = export_subtitles_srt_and_ass(
                    segs,
                    final_p.replace(".mp4", ".srt"),
                    final_p.replace(".mp4", ".ass"),
                    style_key=BATCH_SUBTITLE_STYLE,
                    play_res_x=vinfo["width"],
                    play_res_y=vinfo["height"],
                )
                sub_p = final_p.replace(".mp4", "_subtitled.mp4")
                burn_ass_subtitles_into_video(final_p, ass_p, sub_p)
                final_p = sub_p
        print(f"     -> Saved: {final_p}")
        display(FileLink(final_p))
    except Exception as e:
        print(f"  [Error] {os.path.basename(vp)}: {e}")
        traceback.print_exc()
